## 00 - Data Preparation

In [6]:
import glob
import unicodedata
from pathlib import Path

import pandas as pd

In [4]:
DATA_DIR = Path('../data')

In [5]:
abstract_folder_path = DATA_DIR / 'Abstracts'
memoire_path = DATA_DIR / 'memoires_actuariat.xlsx'

# Load files into a dictionary
files_dict = {}

for file_path in glob.glob(f"{abstract_folder_path}/*"):
    if Path(file_path).is_file():
        filename = Path(file_path).name
        with open(file_path, encoding='utf-8') as f:
            files_dict[filename[:-4]] = f.read()

print(f"Loaded {len(files_dict)} files")

df_memoire = pd.read_excel(memoire_path)
df_memoire["titre"] = df_memoire["titre"].str.strip()

df_abstract = pd.DataFrame(list(files_dict.items()), columns=['Filename', 'Content'])
df_abstract["Filename"] = df_abstract["Filename"].str.strip()

Loaded 3294 files


In [ ]:
def clean_string(text):
    text = unicodedata.normalize('NFC', text)
    text = text.strip()
    return text

In [ ]:
df_memoire["titre"] = df_memoire["titre"].apply(clean_string)
df_abstract["Filename"] = df_abstract["Filename"].apply(clean_string)

df_merged = df_memoire.merge(df_abstract, left_on='titre', right_on='Filename', how='left')

# Cleaning up the merged DataFrame
df_merged_clean = df_merged.dropna(subset=['Content'])
del df_merged_clean["Filename"]
df_merged_clean.rename(columns={"auteur": "author", "annee": "year", 
                                "titre": "title", "titre_full": "full_title",
                                "Content": "content",
                                "entreprise": "company"}, inplace=True)

del df_merged_clean["PDF"]
del df_merged_clean["ABS"]

In [ ]:
df_merged_clean.to_parquet(DATA_DIR / "abstracts.parquet", index=False)